In [1]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import numpy as np
import sys 
from pathlib import Path
import random
from pprint import pp
import time

In [8]:
ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
from src.chunks import read_pages
from src.config import CHROMA_DIR,EMBED_MODEL,COLLECTION_NAME

In [3]:
emb = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    encode_kwargs={"normalize_embeddings": True},
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [ ]:
phrases = [
    "Le ratio CET1 du Groupe s'établit à 12,9 %.",
    "UBS's CET1 capital ratio was 14.4% at year end.",
    "Le Conseil d'administration s'est réuni 9 fois en 2025.",
]
question = "Quel est le ratio CET1 ?"
#question = "Quel est le niveau de solvabilité de la banque ?"
vecteurs = np.array(emb.embed_documents(phrases))
vec_question = np.array(emb.embed_query(question))


print(vecteurs.shape)          # (3, 1024) : 3 phrases, 1 024 nombres chacune
print(vecteurs @ vec_question)     # similarité de chaque phrase avec la question
print(vecteurs @ vecteurs.T)   # similarité de chaque phrase avec les autres

In [ ]:
chunks = read_pages(ROOT / 'data/processed/chunks.jsonl')
extrait_chunks = random.sample(chunks, 50)


## Creation de collection chroma sur extrait

In [ ]:
collect_test = Chroma(
    collection_name = 'test_pages',
    embedding_function = emb,
    persist_directory = str(ROOT /'data/chroma_test'),
    collection_configuration = {"hnsw": {"space" : "cosine"}},
)

In [ ]:
collect_test.add_documents(documents=extrait_chunks,
                      ids= [c.metadata['chunk_id'] for c in extrait_chunks])

In [ ]:
collect_test.similarity_search_with_score(query=question, k=3,)

In [ ]:
pp(collect_test.similarity_search_with_score(query=question, k=3,)[0][0].page_content)

In [ ]:
embedding_func = HuggingFaceEmbeddings(model= EMBED_MODEL, encode_kwargs = {"normalize_embeddings": True})


In [6]:
def build_chroma(chunks, taille_lot=500):
    collect = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embedding_func,
        persist_directory = str(CHROMA_DIR),
        collection_configuration = {"hnsw": {"space" : "cosine"}}
    )
    print(f'{len(chunks)} chunks a indexer dans {CHROMA_DIR}')
    for i in range(0,len(chunks), taille_lot):
        chunks_petit = chunks[i:i+taille_lot]
        collect.add_documents(documents=chunks_petit, ids=[c.metadata['chunk_id'] for c in chunks_petit])
        print(f'{i+len(chunks_petit)}/{len(chunks)} chunks ont ete ajoutes dans index Chroma.')
    return collect

In [ ]:
start_time = time.perf_counter()
build_chroma(extrait_chunks)
print(time.perf_counter()-start_time)

In [ ]:
quot,rest = divmod(3567,20)
quot,rest

In [ ]:
20*178

In [10]:
from src.indexing import get_embeddings, build_chroma

In [4]:
chunks = read_pages(ROOT / 'data/processed/chunks.jsonl')

In [11]:
start_time = time.perf_counter()
build_chroma(chunks)
print(time.perf_counter()-start_time)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

5840 chunks a indexer dans /home/yanis/projects/rapFin_assistant/data/index/chroma
500/5840 chunks ont ete ajoutes dans index Chroma.
1000/5840 chunks ont ete ajoutes dans index Chroma.
1500/5840 chunks ont ete ajoutes dans index Chroma.
2000/5840 chunks ont ete ajoutes dans index Chroma.
2500/5840 chunks ont ete ajoutes dans index Chroma.
3000/5840 chunks ont ete ajoutes dans index Chroma.
3500/5840 chunks ont ete ajoutes dans index Chroma.
4000/5840 chunks ont ete ajoutes dans index Chroma.
4500/5840 chunks ont ete ajoutes dans index Chroma.
5000/5840 chunks ont ete ajoutes dans index Chroma.
5500/5840 chunks ont ete ajoutes dans index Chroma.
5840/5840 chunks ont ete ajoutes dans index Chroma.
8332.267400797005
